# Lab03 — mudança e persistência condicionais dos betas
**Além do Beta · CMP609 · 23/10/2026**

O Lab02Y executado pelo autor foi conferido: nove códigos em sequência, oito testes, zero exceções, 14 exports autenticados e três gráficos com pixels coincidentes. Este caderno implementa o plano temporal já registrado, nas amostras completas: 80 ativos em 2016–2025 e 117 em 2021–2025, dez setores, sem seleção anual, pesos por empresa ou agregação de classes.

**Os modelos usam retornos diários observados, não preços ou retornos esperados previstos.** Os anos e janelas organizam esses pares diários para estimar os coeficientes. Benchmark: Ibovespa B3. Os 80 comuns pertencem aos 117 e reutilizam seus resultados nos mesmos recortes.

## Executar um lote para todas as amostras
Execute todas as células e autorize o Drive; ao fim, salve o notebook com as saídas. O runtime leve é autenticado e reutiliza os ZIPs efetivos X (~19 MB) e Y (~2,6 MB). Caches locais com hash correto são reutilizados; não há download de cotações ou históricos de mercado.

**Alcance condicional:** o gate de homologação global do plano original continua não superado. Executam-se os contrastes sobre a referência observada N e sensibilidades explícitas, sem reclassificá-los como conclusão confirmatória independente. Agregados e estimativas anuais já haviam sido inspecionados; não se alega pré-registro independente. Permanecem sete ausências na década (cinco ABCB4/UGPA3 + duas ANIM3), Light conciliada, 0/197 históricos homologados, zero líquidos RENDIMENTO no corte e vieses de seleção retrospectiva/sobrevivência.

In [ ]:
from pathlib import Path
import os,sys,io,json,hashlib,zipfile,tempfile,datetime,subprocess,platform
import numpy as np
import pandas as pd
from IPython.display import display,Image
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
FOLDER_ID='1XRcIHVAoBVmX9sDWvAuNDCJ5dKEKqedd'
RUNTIME_ID='1sy_YJ93ZTe0K0jm7oCPbVmerFDl1gan5'
MANIFEST_ID='15zAkDSkB24pHjr2cBYRnrPOEiRD90-6_'
EXPECTED_MANIFEST_SHA='228314b53f38bfb4eb34c7a57b00bf3fed1f60908dc060bcd1cb534cb1d62f1d'
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
    credentials,_=google.auth.default()
    drive_service=build('drive','v3',credentials=credentials,cache_discovery=False)
    def download_bytes(file_id):
        stream=io.BytesIO();request=MediaIoBaseDownload(stream,drive_service.files().get_media(fileId=file_id));done=False
        while not done: _,done=request.next_chunk()
        return stream.getvalue()
    manifest_bytes=download_bytes(MANIFEST_ID);package_bytes=download_bytes(RUNTIME_ID)
    CACHE=Path('/content/alem_beta_y_cache');CACHE.mkdir(exist_ok=True)
else:
    PACKAGE_DIR=Path(os.environ['LAB03_PACKAGE_DIR'])
    manifest_bytes=(PACKAGE_DIR/'lab03_runtime_manifest.json').read_bytes()
    package_bytes=(PACKAGE_DIR/'lab03_runtime_v1.zip').read_bytes()
assert hashlib.sha256(manifest_bytes).hexdigest()==EXPECTED_MANIFEST_SHA,'Manifesto divergente'
runtime=json.loads(manifest_bytes)
assert hashlib.sha256(package_bytes).hexdigest()==runtime['package_sha256'],'Runtime divergente'
REPO=Path(tempfile.mkdtemp(prefix='alem_beta_lab03_'))
with zipfile.ZipFile(io.BytesIO(package_bytes)) as z:
    assert len(z.namelist())==len(set(z.namelist())) and set(z.namelist())==set(runtime['files'])
    for name,item in runtime['files'].items():
        p=Path(name);assert not p.is_absolute() and '..' not in p.parts
        raw=z.read(name);assert len(raw)==item['bytes'] and hashlib.sha256(raw).hexdigest()==item['sha256']
        target=REPO/p;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
lock=json.loads((REPO/'data/lab03/inputs_manifest.json').read_text())
archives={}
for key in ['x','y']:
    if IN_COLAB:
        path=CACHE/(lock[key+'_results_sha256']+'.zip')
        if not path.exists():
            raw=download_bytes(lock[key+'_results_drive_id'])
            assert hashlib.sha256(raw).hexdigest()==lock[key+'_results_sha256']
            path.write_bytes(raw)
    else:path=Path(os.environ['LAB03_'+key.upper()+'_ARCHIVE'])
    assert hashlib.sha256(path.read_bytes()).hexdigest()==lock[key+'_results_sha256'],'Cache divergente: '+key
    archives[key]=path
    os.environ['LAB03_'+key.upper()+'_ARCHIVE']=str(path)
X_ARCHIVE,Y_ARCHIVE=archives['x'],archives['y']
sys.path[:0]=[str(REPO/'src'),str(REPO)]
ENV=dict(os.environ,PYTHONPATH=str(REPO/'src')+os.pathsep+str(REPO))
OUT=Path(os.environ.get('LAB03_OUTPUT',str(REPO/'resultados_lab03')))
print('Runtime e ZIPs X/Y autenticados. Ambiente:', 'Colab' if IN_COLAB else 'preparação local; não é conferência Colab')
print('Saídas:',OUT)

## 1. Proveniência, protocolo e configurações
O protocolo original `data/lab02/plano_temporal_v1.json` permanece intacto. O adendo `data/lab03/plano_execucao_v1.json` fixa a execução condicional, dez configurações e o alcance das sensibilidades. O principal é N/log/HAC5; alternativas são HAC0/10/20 em N, retornos simples em N, quatro variantes de caixa X e a união de janelas de eventos já definidas. Isso não é um fatorial completo nem uma escolha pelo menor p-valor.

Retornos simples transformam ativo e mercado. A variante de eventos une ±5 pregões e o quarto trimestre da Light; não remove anos inteiros. A sensibilidade temporal de negociação B3 nas duas extremidades depende de dados de negociação autenticados e não foi executada aqui. Não promover cenários de caixa a reconstrução global dos fatores do fornecedor.

In [ ]:
receipt=json.loads((REPO/'data/lab02/validacao_lab02y_colab_20261023.json').read_text())
assert receipt['colab_execution_verified'] and receipt['tests']==8 and receipt['exceptions']==0
assert receipt['authenticated_exports']==14
plan=json.loads((REPO/'data/lab03/plano_execucao_v1.json').read_text())
print('Y conferido; execução UTC:',receipt['execution_record']['executed_utc'])
display(pd.DataFrame(plan['configurations']))

## 2. Mudança de magnitude: duas metades dos mesmos 80 ativos
Fixe $D_t=1$ se o mercado sobe e $Q_t=1$ em 2021–2025. Nos pares válidos não neutros, o modelo de oito parâmetros é

$$
r_{it}=a+gD_t+bm_t+\Delta D_tm_t+Q_t(\eta_0+\eta_1D_t+\eta_2m_t+\theta D_tm_t)+\epsilon_t.
$$

$\theta=\Delta\beta_{2021-2025}-\Delta\beta_{2016-2020}$. Os interceptos e inclinações variam por metade e regime. HAC Bartlett5 usa os escores no calendário original, correção $n/(n-8)$ e a covariância conjunta, inclusive na fronteira entre metades. **Não se somam variâncias como se fossem independentes.** O teste bilateral usa aproximação normal e IC95% marginal.

Cada metade exige os pisos do núcleo: 200 pares, 60 por regime, 95% de cobertura e variância de mercado ≥10⁻¹². Não rejeitar θ=0 não demonstra estabilidade ou equivalência. A data divisória vem das janelas fixas; não se procura a maior quebra nem se identifica causalidade.

## 3. Persistência de direção: todos os anos, nas duas direções
Para cada ativo/janela, estime Δβ com os retornos diários de cada ano planejado: dez na década ou cinco recentes. Para a direção positiva, $H_0$ é existir ao menos um ano com $\Delta\beta_y\le0$; a alternativa exige $\Delta\beta_y>0$ em todos. Para a negativa, inverta os sinais.

Use $p_+=\max_y[1-\Phi(\widehat\Delta_y/SE_y)]$ e $p_-=\max_y\Phi(\widehat\Delta_y/SE_y)$. A interseção–união (IUT) exige evidência em **todos** os anos. Sob a nula, pelo menos um componente é verdadeiro, e o evento “todos rejeitam” está contido na rejeição desse componente; o princípio não exige independência anual. A validade aplicada ainda depende da aproximação HAC de cada componente, e o teste pode ser conservador/de baixo poder. Fonte metodológica já registrada: Berger e Hsu (1996), seção 3; não se transporta calibração biomédica para retornos.

As duas direções são testadas; não se escolhe apenas o sinal observado. Um ano insuficiente ou com variância degenerada torna a hipótese não estimável, sem remover ano/ativo. Direção em todos os anos observados não é magnitude constante, previsão ou persistência futura.

## 4. Família de 474 e testes de implementação
São 80 contrastes de magnitude + duas direções × (80 década + 117 recentes) = **474 hipóteses**. Holm a 5% é aplicado a essa família por configuração; hipóteses não estimáveis preservam posições e não são rejeitadas. Os 80 comuns reutilizam as direções dos 117. Não se alega controle conjunto com as 197 hipóteses estáticas do Lab02 ou com seleção entre configurações.

Os testes de código verificam dados sintéticos com mudança conhecida, regressões separadas, correção conjunta n/(n−8), covariância na fronteira, calendário com ausências/neutros, IUT nas duas direções, anos insuficientes, degeneração, 474 posições, reutilização dos 80, janelas 252/21 e autenticação/ausências/Light.

In [ ]:
import unittest
suite=unittest.TestLoader().discover(str(REPO/'tests'),pattern='test_lab03.py')
test_result=unittest.TextTestRunner(verbosity=1).run(suite)
assert test_result.wasSuccessful() and test_result.testsRun==14
print('14 testes Lab03 aprovados.')

## 5. Executar o lote temporal completo
As janelas móveis têm 252 pregões, passo de 21; 504 pregões é sensibilidade. Aplicam-se os mesmos pisos e preserva-se o calendário, sem concatenar observações restantes ou preencher as sete ausências. Todas as janelas/status/ICs são exportados, inclusive insuficientes.

Janelas sobrepostas compartilham retornos: não contar seus sinais/rejeições como testes independentes nem inferir persistência pela sobreposição de ICs. As janelas móveis usam somente N/log/HAC5 e não compõem a família474. As dez configurações se aplicam aos modelos anuais/metades/IUT; não se alega janela móvel de todas as variantes de caixa.

A execução imprime os avanços por configuração e por grupo de janelas. Aguarde a conclusão antes de seguir.

In [ ]:
process=subprocess.Popen([sys.executable,str(REPO/'scripts/run_lab03.py'),'--x-archive',str(X_ARCHIVE),'--y-archive',str(Y_ARCHIVE),'--output',str(OUT)],env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:print(line,end='',flush=True)
assert process.wait()==0,'Falha na execução temporal'
metrics=json.loads((OUT/'metricas.json').read_text())
assert metrics['annual_rows']==17850 and metrics['joint_half_rows']==800
assert metrics['temporal_family_rows']==4740 and metrics['rolling_rows']==32708
assert not metrics['confirmatory_global_data_gate_passed']
display(pd.DataFrame([metrics]))

## 6. Resultados: estimativas condicionais e alternativas predefinidas
As contagens abaixo são obtidas dos arquivos completos. A configuração principal foi fixada antes desta execução. Não se escolhe uma alternativa por apresentar mais rejeições.

Ausência de rejeições não prova equivalência, estabilidade, ausência de assimetria ou ausência de dependência temporal. A IUT é exigente e a família474 reduz poder. Uma mediana de θ é descrição entre ativos, não teste de mudança populacional ou peso de empresa. Resultados continuam sujeitos à mensuração dos retornos, cobertura dos eventos e seleção retrospectiva.

In [ ]:
summary=pd.read_csv(OUT/'summary.csv')
display(summary)
family=pd.read_csv(OUT/'temporal_family.csv')
primary=family[family.configuration.eq('N_log_HAC5')]
assert len(primary)==474
print('Hipóteses abaixo de 5% após Holm na configuração principal:',int(primary.reject_005_conditional.sum()))
display(primary.head(10))

## 7. Quatro figuras e exportação vetorial
Veja todos os 80 contrastes com IC marginal conjunto, mapas anuais dos 80 e 117 e janelas de WEGE3 (exemplo já usado no Lab02). Cada célula anual é estimada com os pares diários do ano; nenhuma regressão usa um único retorno anual como amostra.

Cor uniforme ou ICs sobrepostos não demonstram persistência. Os 80 comuns nas janelas reutilizam os 117; o gráfico repetido não acrescenta evidência independente. Todos os ativos estão nos CSVs, e os mapas anuais exibem os dez anos completos da amostra década. PDFs vetoriais são exportados; uma versão compacta do gráfico móvel é reservada ao artigo.

In [ ]:
subprocess.run([sys.executable,str(REPO/'scripts/plot_lab03.py'),'--output',str(OUT)],check=True,env=ENV)
for name in ['lab03_theta80','lab03_anual_decada','lab03_anual_recente','lab03_rolling_wege3']:
    display(Image(filename=str(OUT/'figures'/(name+'.png')),width=1200))

## 8. Insuficiências e igualdade no período comum
Retirar o quarto trimestre da Light deixa 190 pares em 2024, abaixo do piso anual200. A linha permanece com estimativa ausente nessa sensibilidade; as direções que dependem desse ano não são estimáveis e continuam reservando lugares na família. Não reduzir o piso ou retirar o ano para criar persistência.

Os 1.600 resultados comuns de direção (160 por configuração) são reutilizados dos 117; os anos comuns também coincidem. As 32.708 linhas móveis cobrem todos os ativos previstos nas janelas disponíveis. Não há novas exclusões de ativos ou classes.

In [ ]:
annual=pd.read_csv(OUT/'annual.csv')
display(annual[annual.status_conditional.ne('ok')][['configuration','scenario','ticker','year','n_pairs','n_plus','n_minus','status_conditional']])
common=pd.read_csv(OUT/'common_directions.csv')
assert len(common)==1600
display(pd.read_csv(OUT/'rolling.csv').groupby(['scenario','width']).size().rename('linhas').reset_index())

## 9. Conferência independente
O verificador independente utiliza regressões/momentos e escores do calendário, sem importar os módulos de produção. Confere todas as linhas anuais, 800 modelos conjuntos, família474 por configuração, 1.600 direções comuns e todas as 32.708 linhas móveis. A aplicação de Holm e a combinação anual são recalculadas, sem usar flags como prova.

A verificação também é extensa e imprime avanços. Valida implementação e aritmética, não resolve o gate global nem certifica aproximações assintóticas, equivalência ou persistência futura.

In [ ]:
process=subprocess.Popen([sys.executable,str(REPO/'scripts/validate_lab03_outputs.py'),'--x-archive',str(X_ARCHIVE),'--output',str(OUT)],env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
json_lines=[]
for line in process.stdout:
    if line.startswith('Conferência independente:'):print(line,end='',flush=True)
    else:json_lines.append(line)
assert process.wait()==0,'Falha na conferência independente'
independent=json.loads(''.join(json_lines))
assert independent['annual_rows_checked']==17850 and independent['joint_models_checked']==800
assert independent['family_slots_checked']==4740 and independent['rolling_rows_checked']==32708
display(pd.DataFrame([independent]))

## 10. Salvar e preparar a versão científica
O ZIP final inclui resultados completos, figuras, manifesto e registro do ambiente. As matrizes X/Y não são duplicadas. Salve também o notebook com as saídas e avise após a conclusão. Não repetir X/Y/W/S.

A execução deste notebook será conferida separadamente; a preparação local não valida automaticamente o Colab. O próximo trabalho é consolidar o artigo de submissão com os resultados e limitações sustentados, para 27/10. Prorrogação até 03/11 permanece não confirmada. A investigação documental global continua aberta e não se transforma em concluída por aprovar testes de execução.

In [ ]:
record=dict(stage='Lab03',environment='Colab' if IN_COLAB else 'local preparation',executed_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,tests_run=test_result.testsRun,tests_successful=test_result.wasSuccessful(),independent_check=independent,input_x_sha256=lock['x_results_sha256'],input_y_sha256=lock['y_results_sha256'],temporal_tests_computed=True,interpretation='conditional_on_observed_reference',global_homologated=0,confirmatory_global_data_gate_passed=False,equivalence_tested=False,future_persistence_tested=False)
(OUT/'registro_execucao.json').write_text(json.dumps(record,indent=2)+'\n')
hashes={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='manifesto_exports.json'}
(OUT/'manifesto_exports.json').write_text(json.dumps(hashes,indent=2)+'\n')
archive=OUT.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,str(p.relative_to(OUT)))
print('ZIP:',archive,'SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
if IN_COLAB:
    name='resultados_lab03_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip'
    uploaded=drive_service.files().create(body={'name':name,'parents':[FOLDER_ID]},media_body=MediaFileUpload(str(archive),mimetype='application/zip',resumable=True),fields='id,webViewLink').execute()
    print('Resultados salvos no Drive:',uploaded.get('webViewLink','https://drive.google.com/file/d/'+uploaded['id']+'/view'))
else:print('Preparação local concluída; não é conferência Colab.')